## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 52 samples and 13410 columns.
First few rows of the data:


,Overall survival(month),Survival state,Sample,TP53 Signture Score,TP53 Signture Group,TP53 Mutation,Gender,Age,Tumor_size,Grade,...,ACACA,TMEM191B,C11orf98,MRM1,H2AC17,H3C10,MRPL45,PAGR1,TMEM265,ADORA3
0,49.0,0,BC1,0.645576,Low,TP53_WT,Man,unknown,unknown,unknown,...,13.088213,2.154999,15.261451,2.467785,0.506301,0.019989,23.010832,18.444067,5.970439,0.846116
1,49.0,0,BC2,1.350214,High,TP53_WT,Man,Young,Small,HG,...,8.068462,1.326360,18.845911,5.124256,0.227321,0.750467,33.204845,15.243621,0.419335,1.226338
2,11.0,1,BC3,0.466355,Low,TP53_WT,Female,Old,Small,HG,...,15.422922,0.370062,19.831106,4.689168,0.359574,0.051241,28.186981,15.683324,1.816877,0.302842
3,2.0,1,BC4,0.315740,Low,TP53_WT,Man,Old,unknown,HG,...,9.595050,1.313457,30.044163,7.445334,0.418230,1.361837,37.776566,26.574846,1.288730,0.508015
4,27.0,1,BC5,0.285227,Low,TP53_WT,Female,Young,Large,HG,...,8.550369,0.903143,65.044716,10.862432,3.148769,3.156988,27.455772,52.573322,3.091669,1.063593


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'Sample'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
